In [1]:
import os
import imageio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML
import matplotlib.patches as patches
import warnings
warnings.filterwarnings('ignore')

C:\Users\tanni\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


In [2]:
df= pd.read_csv("traffic.csv")

In [3]:
df

,DateTime,Junction,Vehicles,ID
0,2015-11-01 00:00:00,1,15,20151101001
1,2015-11-01 01:00:00,1,13,20151101011
2,2015-11-01 02:00:00,1,10,20151101021
3,2015-11-01 03:00:00,1,7,20151101031
4,2015-11-01 04:00:00,1,9,20151101041
...,...,...,...,...
48115,2017-06-30 19:00:00,4,11,20170630194
48116,2017-06-30 20:00:00,4,30,20170630204
48117,2017-06-30 21:00:00,4,16,20170630214
48118,2017-06-30 22:00:00,4,22,20170630224


In [4]:
df.head()

,DateTime,Junction,Vehicles,ID
0,2015-11-01 00:00:00,1,15,20151101001
1,2015-11-01 01:00:00,1,13,20151101011
2,2015-11-01 02:00:00,1,10,20151101021
3,2015-11-01 03:00:00,1,7,20151101031
4,2015-11-01 04:00:00,1,9,20151101041


In [5]:
df['DateTime'] = pd.to_datetime(df['DateTime'])
df['year'] = df['DateTime'].dt.year
df['month'] = df['DateTime'].dt.month
df['date'] = df['DateTime'].dt.day
df['hour'] = df['DateTime'].dt.hour
df['weekday'] = df['DateTime'].dt.day_name()
df['Meridiem'] = df['hour'].apply(lambda x: 'AM' if x < 12 else 'PM')

In [6]:
df = df.drop(columns = ["DateTime", "ID"])

In [7]:
cat_col = df.select_dtypes(include = "object")
for col in cat_col:
    print(f"{col} has {df[col].unique()}")

weekday has <ArrowStringArray>
['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
Length: 7, dtype: str
Meridiem has <ArrowStringArray>
['AM', 'PM']
Length: 2, dtype: str


In [8]:
weekday_map = {
    'Sunday': 0, 'Monday': 1, 'Tuesday': 2, 'Wednesday': 3,
    'Thursday': 4, 'Friday': 5, 'Saturday': 6
}
df['weekday'] = df['weekday'].map(weekday_map)

df['Meridiem'] = df['Meridiem'].map({'AM': 0, 'PM': 1})

In [9]:
df.corr()["Vehicles"].drop("Vehicles")*100

Junction   -61.378721
year        22.125305
month       -2.272345
date         2.789498
hour        21.993767
weekday      1.056369
Meridiem    23.298452
Name: Vehicles, dtype: float64

In [10]:
filtered_df = df[
    (df['Junction'] == 2) &
    (df['year'] == 2016) &    # Replace with your desired year
    (df['month'] == 5) &      # Replace with your desired month
    (df['date'] == 15) &      # Replace with your desired date
    (df['hour'] == 13) &       # Replace with your desired hour (24-hour format)
    (df['Meridiem'] == 1)     # 0 for AM, 1 for PM – only include if needed
]

In [11]:
total_vehicles = filtered_df['Vehicles'].sum()
print(f"Total vehicles: {total_vehicles}")

Total vehicles: 10


In [12]:
specific_date = (df['year'] == 2016) & (df['month'] == 5) & (df['date'] == 15) & (df['Meridiem'] == 0)
hours = sorted(df[specific_date]['hour'].unique())

os.makedirs("frames", exist_ok=True)

def plot_dots(ax, start, end, count, color='blue', max_dots=50, jitter=0.02):
    count = min(count, max_dots)
    if start[0] == end[0]:
        x_points = np.full(count, start[0]) + np.random.uniform(-jitter, jitter, count)
        y_points = np.random.uniform(min(start[1], end[1]), max(start[1], end[1]), count)
    elif start[1] == end[1]:
        x_points = np.random.uniform(min(start[0], end[0]), max(start[0], end[0]), count)
        y_points = np.full(count, start[1]) + np.random.uniform(-jitter, jitter, count)
    else:
        x_points = np.random.uniform(min(start[0], end[0]), max(start[0], end[0]), count)
        y_points = np.random.uniform(min(start[1], end[1]), max(start[1], end[1]), count)

    ax.scatter(x_points, y_points, color=color, s=50, alpha=0.8, edgecolor='black')

filenames = []
for hour in hours:
    filtered_df = df[specific_date & (df['hour'] == hour)]
    vehicles_by_junction = filtered_df.groupby('Junction')['Vehicles'].sum().to_dict()
    all_junctions = [1, 2, 3, 4]
    for j in all_junctions:
        vehicles_by_junction.setdefault(j, 0)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.set_facecolor('#e0e0e0')

    road_width = 0.3
    ax.add_patch(patches.Rectangle((0.35, 0), road_width, 1, color='gray'))
    ax.add_patch(patches.Rectangle((0, 0.35), 1, road_width, color='gray'))

    for offset in [-0.05, 0.05]:
        ax.plot([0.5 + offset, 0.5 + offset], [0, 1], color='white', linestyle='--', linewidth=2)
        ax.plot([0, 1], [0.5 + offset, 0.5 + offset], color='white', linestyle='--', linewidth=2)

    plot_dots(ax, (0.4, 0.7), (0.4, 0.9), vehicles_by_junction[1], 'red')
    plot_dots(ax, (0.6, 0.6), (0.8, 0.6), vehicles_by_junction[2], 'green')
    plot_dots(ax, (0.2, 0.4), (0.4, 0.4), vehicles_by_junction[3], 'blue')
    plot_dots(ax, (0.6, 0.2), (0.6, 0.4), vehicles_by_junction[4], 'orange')

    ax.text(0.15, 0.85, 'J1', fontsize=14, fontweight='bold', color='red')
    ax.text(0.1, 0.8, f"Vehicles: {vehicles_by_junction[1]}", fontsize=12, color='red')
    ax.text(0.8, 0.85, 'J2', fontsize=14, fontweight='bold', color='green')
    ax.text(0.75, 0.8, f"Vehicles: {vehicles_by_junction[2]}", fontsize=12, color='green')
    ax.text(0.15, 0.2, 'J3', fontsize=14, fontweight='bold', color='blue')
    ax.text(0.1, 0.15, f"Vehicles: {vehicles_by_junction[3]}", fontsize=12, color='blue')
    ax.text(0.8, 0.2, 'J4', fontsize=14, fontweight='bold', color='orange')
    ax.text(0.75, 0.15, f"Vehicles: {vehicles_by_junction[4]}", fontsize=12, color='orange')

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    plt.title(f"Vehicle Counts at Junctions on May 15, 2016 – Hour: {hour}:00", fontsize=16)
    plt.tight_layout()

    filename = f"frames/frame_{hour:02d}.png"
    plt.savefig(filename)
    filenames.append(filename)
    plt.close()

In [13]:
with imageio.get_writer("vehicle_progression.gif", mode='I', duration=0.7, loop=0) as writer:
    for filename in filenames:
        image = imageio.imread(filename)
        writer.append_data(image)

In [14]:
HTML('<img src="vehicle_progression.gif" width="600">')